# NC-03: Spatial Analysis

Loads the exported graph, converts it to NetworkX, then computes
standard space-syntax-inspired metrics: degree, closeness, and
betweenness centrality, shortest paths, and community detection.

**Run NC-02 first** to generate the CSV files.

In [ ]:
from pathlib import Path

from topologicpy.Graph import Graph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge

import networkx as nx
import pandas as pd

## 0. Paths

In [ ]:
BASE        = Path(r'E:\softwares-4\graph-ml\assign-04-node-classification')
GRAPHS_PATH = BASE / 'graphs'
print('Graphs folder:', GRAPHS_PATH)

## 1. Load graph from CSV

In [ ]:
graphs = Graph.ByCSVPath(path=str(GRAPHS_PATH))
print('Graphs loaded:', len(graphs))

graph = graphs[0]
vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph)    or []
print('Vertices:', len(vertices))
print('Edges   :', len(edges))

# Show room types on each vertex
print()
print('Room type per node:')
for i, v in enumerate(vertices):
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, 'room_type') or Dictionary.ValueAtKey(d, 'label') or '?'
    print(f'  node {i}: {rt}')

## 2. Convert to NetworkX

Builds an undirected NetworkX graph from the TopologicPy graph vertices
and edges, assigning room-type labels as node attributes.

In [ ]:
G = nx.Graph()

# Build vertex index (TopologicPy vertex -> integer node id)
v_index = {}
for i, v in enumerate(vertices):
    d       = Topology.Dictionary(v)
    rt      = Dictionary.ValueAtKey(d, 'room_type') or ''
    label   = Dictionary.ValueAtKey(d, 'label')
    G.add_node(i, room_type=rt, label=label)
    v_index[i] = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))

# Map coordinates back to index for edge lookup
coord_to_idx = {coord: idx for idx, coord in v_index.items()}

def vertex_id(v):
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    return coord_to_idx.get(key)

for e in edges:
    sv = Edge.StartVertex(e)
    ev = Edge.EndVertex(e)
    si = vertex_id(sv)
    ei = vertex_id(ev)
    if si is not None and ei is not None and si != ei:
        G.add_edge(si, ei)

print('NetworkX graph:')
print('  Nodes:', G.number_of_nodes())
print('  Edges:', G.number_of_edges())
print('  Connected:', nx.is_connected(G))

## 3. Degree centrality

Measures how many rooms each room is directly connected to.

In [ ]:
degree_centrality = nx.degree_centrality(G)

print('Degree centrality (normalised):')
for node, val in sorted(degree_centrality.items(), key=lambda x: -x[1]):
    rt = G.nodes[node].get('room_type', '?')
    print(f'  node {node:2d} ({rt:12s}): {val:.3f}  [degree={G.degree(node)}]')

## 4. Closeness centrality

How quickly a room can reach all other rooms (integration / access efficiency).

In [ ]:
closeness_centrality = nx.closeness_centrality(G)

print('Closeness centrality:')
for node, val in sorted(closeness_centrality.items(), key=lambda x: -x[1]):
    rt = G.nodes[node].get('room_type', '?')
    print(f'  node {node:2d} ({rt:12s}): {val:.3f}')

## 5. Betweenness centrality

How often a room lies on the shortest path between other rooms (choice / movement potential).

In [ ]:
betweenness_centrality = nx.betweenness_centrality(G, normalized=True)

print('Betweenness centrality:')
for node, val in sorted(betweenness_centrality.items(), key=lambda x: -x[1]):
    rt = G.nodes[node].get('room_type', '?')
    print(f'  node {node:2d} ({rt:12s}): {val:.3f}')

## 6. Shortest paths

Pair-wise shortest path lengths between all rooms.

In [ ]:
spl = dict(nx.all_pairs_shortest_path_length(G))

print('Shortest path lengths (node pairs):')
for src in sorted(spl.keys()):
    src_rt = G.nodes[src].get('room_type', '?')
    for dst in sorted(spl[src].keys()):
        if dst <= src:
            continue
        dst_rt = G.nodes[dst].get('room_type', '?')
        d = spl[src][dst]
        print(f'  {src_rt:12s} -> {dst_rt:12s}: {d} hop(s)')

## 7. Community detection

Grouping rooms into spatial communities using the Louvain method.

In [ ]:
try:
    from networkx.algorithms.community import louvain_communities
    communities = louvain_communities(G, seed=42)
    print(f'Communities found: {len(communities)}')
    for i, comm in enumerate(communities):
        room_names = [G.nodes[n].get('room_type', '?') for n in sorted(comm)]
        print(f'  Community {i}: {room_names}')
except ImportError:
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G))
    print(f'Communities found: {len(communities)}')
    for i, comm in enumerate(communities):
        room_names = [G.nodes[n].get('room_type', '?') for n in sorted(comm)]
        print(f'  Community {i}: {room_names}')

## 8. Summary table

In [ ]:
rows = []
for node in sorted(G.nodes()):
    rows.append({
        'node_id'             : node,
        'room_type'           : G.nodes[node].get('room_type', '?'),
        'label'               : G.nodes[node].get('label', -1),
        'degree'              : G.degree(node),
        'degree_centrality'   : round(degree_centrality[node], 4),
        'closeness_centrality': round(closeness_centrality[node], 4),
        'betweenness_centrality': round(betweenness_centrality[node], 4),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# Save to CSV
out_csv = GRAPHS_PATH / 'spatial_analysis.csv'
summary_df.to_csv(out_csv, index=False)
print()
print('Saved to:', out_csv)